# 05 · Análisis de errores: en qué se equivoca cada sistema y por qué

### Qué es este notebook

Los notebooks 02 a 04 dicen **cuánto** acierta cada sistema. Este dice **en qué falla y por qué**. No entrena ni consulta ningún modelo: reutiliza las predicciones ya guardadas en `results/cache/` (E1, E2 y la nube) y las cruza con la anotación de los médicos. Corre en pocos minutos y no gasta nada.

### Las cuatro preguntas

1. **¿Qué tipo de error comete cada sistema?** Omitir una entidad, marcar algo que no es, o marcarla con límites distintos (una palabra de más o de menos).
2. **¿Qué pasa con las palabras mal escritas?** Las derivaciones reales traen errores como "FUSROSEMIDA" o "ISUFICIENCIA". No se simula nada: se mide el ruido que ya trae el corpus.
3. **¿Reconocen la jerga local?** Abreviaturas frecuentes, raras y nunca vistas en entrenamiento.
4. **¿Sirve combinar encoder y LLM?** Si cada uno encuentra cosas que el otro no, un sistema híbrido podría ganar.

### Conexión con el curso

- **U2–U3 (tokenización y encoders):** la tokenización por subpalabras explica por qué el encoder falla con palabras mal escritas.
- **U5–U8 (LLM y *prompting*):** los LLM marcan de más porque no conocen la convención de anotación del corpus.
- **U4 (modelos híbridos):** combinación de un encoder con un LLM.

> **Ojo con las salidas:** este notebook muestra ejemplos de texto del corpus. Antes de hacer *commit*, limpiar las salidas (*Clear All Outputs*) o correr `git restore notebooks/`.

In [ ]:
from pathlib import Path
import sys, json
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from datos.cwlc import cargar_split
from evaluacion import metricas
from analisis import errores as A
from reproducibilidad import verificar_huellas_datos, huella_codigo

carpeta = RAIZ / "data" / "raw" / "cwlc"
train, dev, test = (cargar_split(carpeta, s) for s in ["train", "dev", "test"])
ok, difs = verificar_huellas_datos(carpeta, "cwlc")
assert ok, f"¡Los datos cambiaron! {difs}"
gold = A.gold_por_tipo(test)
por_id = {o.id: o for o in test}
pd.set_option("display.precision", 3)
print(f"test = {len(test)} oraciones · huellas verificadas")

## 1. Cargar las predicciones guardadas

- **E1:** 2 encoders × 3 semillas (los `.pkl` que guardó el notebook 02).
- **E2 y nube:** las respuestas guardadas de cada LLM, convertidas a entidades con el mismo localizador de E2. Una corrida que todavía no cubre todo el test (por ejemplo Gemini mientras avanza por cuota) se deja fuera y se avisa.

In [ ]:
CACHE = RAIZ / "results" / "cache"
SEMILLAS = [42, 43, 44]
ENCODERS = {"E1 español": "espanol_clinico", "E1 chileno": "chileno_clinico",
            "E1 RigoBERTa": "rigoberta_clinico"}   # el último solo si ya se corrió el notebook 02b
LLMS = {
    "Llama 8B sin ejemplos": "e2_completo/llama3.1-8b_zero_shot",
    "Llama 8B ej. parecidos": "e2_completo/llama3.1-8b_few_shot_dinamico",
    "Qwen 7B sin ejemplos": "e2_completo/qwen2.5-7b_zero_shot",
    "Qwen 7B ej. parecidos": "e2_completo/qwen2.5-7b_few_shot_dinamico",
    "GPT-5.2 sin ejemplos": "e2nube_completo/openai_gpt-5.2_zero_shot",
    "GPT-5.2 ej. parecidos": "e2nube_completo/openai_gpt-5.2_few_shot_dinamico",
    "Gemini 3.1 Pro sin ejemplos": "e2nube_completo/gemini_gemini-3.1-pro-preview_zero_shot",
    "Gemini 3.1 Pro ej. parecidos": "e2nube_completo/gemini_gemini-3.1-pro-preview_few_shot_dinamico",
}

e1 = {}  # (nombre, semilla) -> {tipo: [entidades]}
for nombre, clave in ENCODERS.items():
    for s in SEMILLAS:
        p = A.cargar_e1(CACHE / "e1_completo", clave, s)
        if p is not None:
            e1[(nombre, s)] = p
ENCODERS = {n: c for n, c in ENCODERS.items() if any((n, s) in e1 for s in SEMILLAS)}
for n in ENCODERS:
    faltan = [s for s in SEMILLAS if (n, s) not in e1]
    if faltan:
        print(f"⚠️ {n}: faltan las semillas {faltan}")
llm = {}
for nombre, ruta in LLMS.items():
    r = A.cargar_llm(CACHE / f"{ruta}.jsonl", test)
    if r is None:
        print(f"⏳ {nombre}: {A.cobertura(CACHE / f'{ruta}.jsonl', test)}/{len(test)} oraciones, queda fuera por ahora")
    else:
        llm[nombre] = r[0]
print(f"E1: {len(e1)} modelos · LLM: {len(llm)} corridas completas")

Control: el F1 estricto recalculado aquí debe coincidir con el de los notebooks 02, 03 y 03b. Si no coincide, algo cambió en las predicciones o en el código.

In [ ]:
filas = []
for nombre in ENCODERS:
    for t in A.TIPOS:
        v = [metricas.contar(gold[t], e1[(nombre, s)][t]).f1 for s in SEMILLAS if (nombre, s) in e1]
        filas.append({"sistema": f"{nombre} (media 3 semillas)", "tipo": A.NOMBRE_TIPO[t], "F1": np.mean(v)})
for nombre, p in llm.items():
    for t in A.TIPOS:
        filas.append({"sistema": nombre, "tipo": A.NOMBRE_TIPO[t], "F1": metricas.contar(gold[t], p[t]).f1})
tabla_f1 = pd.DataFrame(filas).pivot(index="sistema", columns="tipo", values="F1")[["Enfermedad", "Medicamento", "Abreviatura"]]
tabla_f1.sort_values("Enfermedad", ascending=False)

## 2. Tipos de error

Cada predicción y cada entidad gold no encontrada se clasifica en:

| Categoría | Qué significa | Ejemplo |
|---|---|---|
| correcta | Coincide exacto con el médico | gold "HTA", predicción "HTA" |
| más larga | Contiene a la entidad gold y algo más | gold "PERIODONTITIS CRONICA", predicción "PERIODONTITIS CRONICA MODERADA" |
| más corta | Queda dentro de la entidad gold | gold "CARIES PROFUNDA", predicción "CARIES" |
| desplazada | Se cruza con la gold sin contenerla | poco frecuente |
| espuria | No toca ninguna entidad gold del mismo tipo | marcar "dolor" como enfermedad |
| omitida | Entidad gold que ninguna predicción toca | no ver "DM2" |

Los tres errores de límite (más larga, más corta, desplazada) son los que el F1 laxo perdona.

In [ ]:
def sistemas_semilla(s=42):
    """E1 con una semilla + todos los LLM, para tablas por sistema."""
    d = {f"{n} (s{s})": e1[(n, s)] for n in ENCODERS if (n, s) in e1}
    d.update(llm)
    return d

SIS = sistemas_semilla(42)
tipologia = {}
for t in A.TIPOS:
    tab = pd.DataFrame({n: A.tipologia(gold[t], p[t]) for n, p in SIS.items()}).T.reindex(columns=A.CATEGORIAS).fillna(0).astype(int)
    tipologia[t] = tab
    print(f"\n{A.NOMBRE_TIPO[t]}"); display(tab)

In [ ]:
t = "Disease"
tab = tipologia[t]
fig, ax = plt.subplots(figsize=(10, 0.45 * len(tab) + 1.5))
izq = np.zeros(len(tab))
colores = {"correcta": "#4c78a8", "más larga": "#bab0ac", "más corta": "#d4cfcb", "desplazada": "#e7e3df", "espuria": "#e45756", "omitida": "#f2a541"}
for c in A.CATEGORIAS:
    ax.barh(tab.index, tab[c], left=izq, color=colores[c], label=c)
    izq += tab[c].values
ax.invert_yaxis(); ax.set_xlabel("número de casos (Enfermedad)")
ax.set_title("Los LLM aciertan casi tanto como E1, pero marcan muchas enfermedades que no lo son")
ax.legend(ncol=6, loc="lower center", bbox_to_anchor=(0.5, -0.32), frameon=False)
plt.tight_layout(); plt.show()

### ¿Qué marcan de más?

Una predicción espuria de Enfermedad puede caer sobre otra entidad que el médico sí anotó, pero con otro tipo. Si cae sobre un **hallazgo** (*Finding*: "dolor", "vómitos", "aumento de volumen"), el sistema no está inventando: está usando un criterio distinto al de la guía del corpus.

In [ ]:
esp = pd.DataFrame({n: A.espurias_sobre_otros_tipos(test, "Disease", p["Disease"]) for n, p in SIS.items()}).T.fillna(0).astype(int)
esp = esp[esp.sum().sort_values(ascending=False).index[:6]]
esp["total espurias"] = tipologia["Disease"]["espuria"]
esp["% sobre un hallazgo"] = 100 * esp.filter(like="Finding").sum(1) / esp["total espurias"]
esp.round(1)

In [ ]:
# Las palabras que más marca cada sistema como enfermedad sin serlo (texto del corpus: no subir esta salida)
for n in [k for k in SIS if "chileno" in k] + [k for k in llm if "GPT" in k or "Gemini" in k]:
    gd = {}
    for g in gold["Disease"]:
        gd.setdefault(g[0], []).append(g)
    sobra = [p for p in SIS[n]["Disease"] if not any(p[1] < g[2] and g[1] < p[2] for g in gd.get(p[0], []))]
    print(f"{n}: ", Counter(por_id[p[0]].mencion(p).lower() for p in sobra).most_common(10))

## 3. Palabras mal escritas en las derivaciones reales

**Cómo se detectan.** Una palabra (de 5 letras o más) se marca como probablemente mal escrita si no está en un diccionario general de español ni en train (ni en plural ni con otro género), y está a 1 o 2 letras de distancia de una palabra frecuente de train. Es una regla automática: su lista sobre el test se revisó a mano y se excluyeron las palabras correctas que marcaba por error (`CORRECTAS_REVISADAS` en `src/analisis/errores.py`).

**Por qué importa.** El encoder ve subpalabras: "furosemida" es un solo token, pero "FUSROSEMIDA" se corta en 5 piezas que nunca vio juntas. Los LLM grandes, entrenados con texto de internet lleno de errores, deberían tolerarlo mejor.

In [ ]:
detector = A.DetectorOrtografia(train)
if not A._diccionario_es():
    print("⚠️ falta pyspellchecker (pip install pyspellchecker): la detección marcará más palabras correctas")
mal = A.menciones_mal_escritas(test, detector)
print(Counter(A.NOMBRE_TIPO[e[3]] for e in mal))
# lista para revisar (texto del corpus: no subir esta salida)
pd.DataFrame([{"tipo": A.NOMBRE_TIPO[e[3]], "mención": por_id[e[0]].mencion(e), "palabra": w, "parece": c}
              for e, ws in mal.items() for w, c in ws])

In [ ]:
def recall_mal_bien(t):
    gm = [e for e in gold[t] if e in mal]
    gb = [e for e in gold[t] if e not in mal]
    filas = []
    for n in ENCODERS:
        rm = [A.recall_grupo(gm, e1[(n, s)][t])[1] for s in SEMILLAS if (n, s) in e1]
        rb = [A.recall_grupo(gb, e1[(n, s)][t])[1] for s in SEMILLAS if (n, s) in e1]
        filas.append({"sistema": f"{n} (3 semillas)", "recall bien escritas": np.mean(rb), "recall mal escritas": np.mean(rm),
                      "rango mal escritas": f"{min(rm):.2f}–{max(rm):.2f}"})
    for n, p in llm.items():
        lo, hi = A.ic_recall(gm, p[t])
        filas.append({"sistema": n, "recall bien escritas": A.recall_grupo(gb, p[t])[1],
                      "recall mal escritas": A.recall_grupo(gm, p[t])[1], "rango mal escritas": f"IC {lo:.2f}–{hi:.2f}"})
    df = pd.DataFrame(filas).set_index("sistema")
    df["caída"] = df["recall bien escritas"] - df["recall mal escritas"]
    print(f"{A.NOMBRE_TIPO[t]}: {len(gm)} mal escritas, {len(gb)} bien escritas")
    return df.sort_values("caída")

recall_mal_bien("Disease")

Medicamento tiene muy pocos casos mal escritos (alrededor de 5), así que no se interpreta. La comparación directa más útil es pareada: encoder contra cada LLM, sobre las **mismas** enfermedades mal escritas.

In [ ]:
gm = [e for e in gold["Disease"] if e in mal]
filas = []
for n_llm, p in llm.items():
    for n in ENCODERS:
        d = [A.dif_recall_pareada(gm, p["Disease"], e1[(n, s)]["Disease"]) for s in SEMILLAS if (n, s) in e1]
        filas.append({"LLM": n_llm, "encoder": n, "LLM − encoder (media)": np.mean([x[0] for x in d]),
                      "IC por semilla": " | ".join(f"[{x[1]:+.2f}, {x[2]:+.2f}]" for x in d)})
pd.DataFrame(filas)

## 4. Menciones vistas y nuevas

¿El sistema encuentra la enfermedad porque la **recuerda** (apareció igual en train) o porque la **entiende**? Se repite para todos los sistemas el análisis que el notebook 02 hizo para E1.

In [ ]:
vistas = {t: {A.normalizar(o.mencion(e)) for o in train for e in o.entidades if e[3] == t} for t in A.TIPOS}
def vistas_nuevas(t):
    gv = [e for e in gold[t] if A.normalizar(por_id[e[0]].mencion(e)) in vistas[t]]
    gn = [e for e in gold[t] if A.normalizar(por_id[e[0]].mencion(e)) not in vistas[t]]
    filas = []
    for n in ENCODERS:
        filas.append({"sistema": f"{n} (3 semillas)",
                      "recall vistas": np.mean([A.recall_grupo(gv, e1[(n, s)][t])[1] for s in SEMILLAS if (n, s) in e1]),
                      "recall nuevas": np.mean([A.recall_grupo(gn, e1[(n, s)][t])[1] for s in SEMILLAS if (n, s) in e1])})
    for n, p in llm.items():
        filas.append({"sistema": n, "recall vistas": A.recall_grupo(gv, p[t])[1], "recall nuevas": A.recall_grupo(gn, p[t])[1]})
    print(f"{A.NOMBRE_TIPO[t]}: {len(gv)} vistas, {len(gn)} nuevas")
    return pd.DataFrame(filas).set_index("sistema")
vistas_nuevas("Disease")

## 5. Lenguaje local: abreviaturas según su frecuencia en train

Las abreviaturas son lo más "chileno" del corpus: APS, PCTE, TTO, PZA, IC, "+". Se agrupan según cuántas veces aparece esa misma abreviatura en train: **frecuente** (10 o más), **rara** (1 a 9) y **nunca vista** (0).

In [ ]:
frec = A.frecuencia_en_train(train, "Abbreviation")
grupos = {}
for e in gold["Abbreviation"]:
    grupos.setdefault(A.grupo_frecuencia(frec[A.normalizar(por_id[e[0]].mencion(e))]), []).append(e)
orden = ["frecuente (≥10 en train)", "rara (1–9 en train)", "nunca vista en train"]
filas = []
for n in ENCODERS:
    fila = {"sistema": f"{n} (3 semillas)"}
    for g in orden:
        fila[g] = np.mean([A.recall_grupo(grupos[g], e1[(n, s)]["Abbreviation"])[1] for s in SEMILLAS if (n, s) in e1])
    filas.append(fila)
for n, p in llm.items():
    filas.append({"sistema": n, **{g: A.recall_grupo(grupos[g], p["Abbreviation"])[1] for g in orden}})
abrev = pd.DataFrame(filas).set_index("sistema")
print({g: len(grupos[g]) for g in orden})
abrev

In [ ]:
ax = abrev.plot.barh(figsize=(10, 0.45 * len(abrev) + 1.5), color=["#4c78a8", "#9ecae9", "#f2a541"])
ax.invert_yaxis(); ax.set_xlim(0, 1); ax.set_xlabel("recall")
ax.set_title("El encoder memoriza las abreviaturas frecuentes; los LLM grandes deducen mejor las nuevas")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.3), ncol=3, frameon=False)
plt.tight_layout(); plt.show()
# las abreviaturas nunca vistas más comunes en el test (texto del corpus: no subir esta salida)
print(Counter(por_id[e[0]].mencion(e) for e in grupos["nunca vista en train"]).most_common(20))

## 6. ¿Sirve combinar encoder y LLM?

**Cruce.** Para cada entidad gold: la encuentra el encoder, el LLM, ambos o ninguno. El *recall oráculo* es lo que lograría un sistema perfecto que supiera siempre a quién creerle: es el techo de cualquier combinación.

**Tres híbridos simples** (sin entrenar nada):

| Híbrido | Regla |
|---|---|
| unión | todo lo que marque cualquiera de los dos |
| rescate | lo del encoder + lo del LLM que no toque nada de lo que marcó el encoder |
| rescate dirigido | como rescate, pero el LLM solo aporta menciones con alguna palabra nunca vista en train (posible error de tipeo o sigla nueva) |

Se compara su F1 con el del encoder solo, con bootstrap pareado (3 semillas).

In [ ]:
filas = []
for t in A.TIPOS:
    for n_llm, p in llm.items():
        if "ej. parecidos" not in n_llm and "Gemini" not in n_llm:
            continue  # los mejores LLM de cada familia
        for n in ENCODERS:
            cr = [A.cruce(gold[t], e1[(n, s)][t], p[t]) for s in SEMILLAS if (n, s) in e1]
            fila = {"tipo": A.NOMBRE_TIPO[t], "encoder": n, "LLM": n_llm,
                    "recall encoder": np.mean([c["recall_A"] for c in cr]), "recall oráculo": np.mean([c["recall_oraculo"] for c in cr]),
                    "solo LLM": np.mean([c["solo B"] for c in cr])}
            for nombre, f in [("unión", lambda a, b: A.union(a, b)), ("rescate", lambda a, b: A.rescate_sin_choque(a, b)),
                              ("rescate dirigido", lambda a, b: A.rescate_dirigido(a, b, test, detector.voc))]:
                d = [A.diferencia_pareada(gold[t], f(e1[(n, s)][t], p[t]), e1[(n, s)][t]) for s in SEMILLAS if (n, s) in e1]
                fila[f"ΔF1 {nombre}"] = np.mean([x[0] for x in d])
                fila[f"¿{nombre} significativo?"] = "mejora" if all(x[1] > 0 for x in d) else ("empeora" if all(x[2] < 0 for x in d) else "no")
            filas.append(fila)
hibridos = pd.DataFrame(filas)
hibridos

In [ ]:
# ¿Qué tienen de especial las enfermedades que solo encuentra el LLM?
n_llm = max(llm, key=lambda k: metricas.contar(gold["Disease"], llm[k]["Disease"]).recall)
for n in ENCODERS:
    solo = [g for g in set(gold["Disease"]) if g in set(llm[n_llm]["Disease"]) and g not in set(e1[(n, 42)]["Disease"])]
    print(f"{n} (s42) vs {n_llm}:", {k: (round(v, 2) if isinstance(v, float) else v) for k, v in A.perfil(solo, test, mal, vistas["Disease"]).items()})

## 7. Combinar modelos por votación (ensamble)

Cada semilla se equivoca en cosas algo distintas. Si una entidad solo la marca un modelo, es más probable que sea un error suelto; si la marcan casi todos, es más confiable. La **votación** se queda con lo que marca la mayoría.

**Regla fijada de antemano:** mayoría estricta (2 de 3 semillas; 4 de 6 modelos). No se prueban varios umbrales en el test: elegir el mejor umbral mirando el test sería hacer trampa.

No hay que entrenar nada: se combinan las predicciones ya guardadas. Se compara con cada modelo individual mediante *bootstrap* pareado.

In [ ]:
grupos_voto = {f"{n} (3 semillas)": [e1[(n, s)] for s in SEMILLAS if (n, s) in e1] for n in ENCODERS}
if len(ENCODERS) > 1:
    grupos_voto[f"Todos los encoders ({len(e1)} modelos)"] = list(e1.values())
filas = []
for nombre, mods in grupos_voto.items():
    for t in A.TIPOS:
        v = A.votacion([m[t] for m in mods])
        f = A.f1_con_ic(gold[t], v)
        ind = [metricas.contar(gold[t], m[t]).f1 for m in mods]
        difs = [A.diferencia_pareada(gold[t], v, m[t]) for m in mods]
        filas.append({"ensamble": nombre, "tipo": A.NOMBRE_TIPO[t], "votos mínimos": f"{len(mods) // 2 + 1} de {len(mods)}",
                      "F1 individual (media)": np.mean(ind), "F1 votación": f["f1"], "IC95": f"{f['ic95'][0]:.3f}–{f['ic95'][1]:.3f}",
                      "P": f["precision"], "R": f["recall"],
                      "mejora vs cada modelo": f"{min(d[0] for d in difs):+.3f} a {max(d[0] for d in difs):+.3f}",
                      "significativa vs todos": all(d[1] > 0 for d in difs)})
votos = pd.DataFrame(filas)
votos.round(3)

## 8. Guardar los resultados

Se guardan solo números (sin texto del corpus) en `results/e5_analisis/analisis_errores.json`, con la huella del código.

In [ ]:
salida = {
    "f1_estricto": tabla_f1.round(4).to_dict(),
    "tipologia": {t: tab.to_dict() for t, tab in tipologia.items()},
    "espurias_enfermedad": esp.drop(columns=["% sobre un hallazgo"]).to_dict(),
    "mal_escritas": {"n": dict(Counter(e[3] for e in mal)), "enfermedad": recall_mal_bien("Disease").drop(columns=["rango mal escritas"]).round(4).to_dict()},
    "vistas_nuevas_enfermedad": vistas_nuevas("Disease").round(4).to_dict(),
    "abreviaturas_por_frecuencia": {"n": {g: len(grupos[g]) for g in orden}, "recall": abrev.round(4).to_dict()},
    "hibridos": hibridos.round(4).to_dict(orient="records"),
    "votacion": votos.round(4).to_dict(orient="records"),
    "encoders": list(ENCODERS),
    "sistemas_llm": list(llm), "parciales": [n for n in LLMS if n not in llm],
    "codigo": huella_codigo(),
}
A.guardar(RAIZ / "results" / "e5_analisis" / "analisis_errores.json", salida)
print("guardado · LLM incluidos:", len(llm), "· pendientes:", salida["parciales"])

## 9. Qué se concluye

Valores con E1 (2 encoders × 3 semillas) y las 8 corridas de LLM completas (Gemini con ejemplos parecidos terminó el 4 de noviembre).

- **Los LLM aciertan casi lo mismo que el encoder, pero marcan de más.** En Enfermedad, GPT-5.2 con ejemplos tiene 854 aciertos exactos contra 895 del encoder chileno, pero 286 espurias contra 112. Gemini sin ejemplos llega a 636 espurias.
- **Lo que sobra son hallazgos, no inventos.** En todos los sistemas, entre el 67% y el 88% de las espurias caen sobre un hallazgo anotado ("dolor", "vómitos", "mordida invertida"). La diferencia está en la cantidad: 112 en el encoder chileno, 286 en GPT-5.2 con ejemplos y 636 en Gemini sin ejemplos. Es una diferencia de criterio con la guía del corpus.
- **Las palabras mal escritas castigan al encoder.** Son 31 enfermedades (3%). El encoder baja de 0,84 a cerca de 0,55 de recall; Gemini casi no baja (0,79 a 0,77). En la comparación pareada sobre esas mismas 31 enfermedades, Gemini supera al encoder por 19 a 32 puntos (el límite inferior del IC queda en 0 o apenas sobre 0, según la semilla); GPT-5.2 con ejemplos lo supera por 10 a 23 puntos, sin significancia. Con tan pocos casos, el efecto en el F1 global es chico.
- **El encoder memoriza la jerga; los LLM grandes la deducen.** Abreviaturas frecuentes: encoder 0,98. Nunca vistas (66): encoder 0,60, GPT-5.2 con ejemplos 0,71, Gemini 0,76.
- **Combinar sin más no sirve.** El recall oráculo sube a 0,90–0,92, pero los tres híbridos simples empeoran el F1, porque lo que agrega el LLM trae muchos falsos positivos. Un híbrido útil necesita saber cuándo duda el encoder, o una verificación.
- **Gemini con ejemplos parecidos encuentra tantas enfermedades como el encoder** (899 exactas contra 895 del encoder chileno; recall 0,84 en ambos), pero marca 406 de más, el 92% sobre hallazgos. Toda su desventaja es de precisión, es decir, de criterio de anotación.
- **Votar entre modelos sí mejora el encoder:** con los 6 modelos de E1 y mayoría estricta (4 de 6), Enfermedad sube de 0,815 (mejor encoder, media de semillas) a 0,837, significativo frente a cada modelo; Abreviatura sube a 0,957. La precisión en Enfermedad pasa de 0,81 a 0,86.
- **Para los errores de tipeo,** la mejora con mejor evidencia es reentrenar el encoder con errores simulados (Moradi y Samwald 2022). Queda como recomendación, porque el impacto global es chico.